# N071 · 树的自底向上聚合

**目标：** 从子树返回最小充分信息而非重复遍历。

**先修：** N069, N063。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 高度/深度；平衡；直径；后序；返回值与全局答案。

## 从问题到算法

自底向上聚合把每个子树压缩成父节点需要的少量信息。深度取左右最大值加一；平衡性要求两边深度差不超过一；经过当前节点的最长路径长度是左右高度之和。直径以边计数，单节点为0，不能与高度的节点数混淆。

## 最小实现

**本章接口：** `max_depth(root)`、`is_balanced(root)`、`diameter(root)`

In [1]:
class TreeNode:
    def __init__(self,val=0,left=None,right=None):
        self.val,self.left,self.right=val,left,right

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None: return None
    root=TreeNode(values[0]); queue=deque([root]); i=1
    while queue and i<len(values):
        node=queue.popleft()
        if values[i] is not None: node.left=TreeNode(values[i]); queue.append(node.left)
        i+=1
        if i<len(values):
            if values[i] is not None: node.right=TreeNode(values[i]); queue.append(node.right)
            i+=1
    return root
def _postorder_nodes(root):
    stack=[(root,False)]
    while stack:
        node,done=stack.pop()
        if not node: continue
        if done: yield node
        else:
            stack.append((node,True)); stack.append((node.right,False)); stack.append((node.left,False))

def max_depth(root):
    heights={None:0}
    for node in _postorder_nodes(root): heights[node]=1+max(heights[node.left],heights[node.right])
    return heights[root]

def is_balanced(root):
    heights={None:0}
    for node in _postorder_nodes(root):
        left,right=heights[node.left],heights[node.right]
        if abs(left-right)>1: return False
        heights[node]=1+max(left,right)
    return True

def diameter(root):
    heights={None:0}; best=0
    for node in _postorder_nodes(root):
        left,right=heights[node.left],heights[node.right]
        best=max(best,left+right); heights[node]=1+max(left,right)
    return best

## 正确性、前提与复杂度

任意简单树路径有唯一最高节点；其两段分别落在该节点左右子树。每个最高节点的最优两段就是最大向下高度，因此取全局最大完备。

**代价：** O(n)时间；显式后序与高度字典O(n)辅助空间，避免深树递归溢出。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([1,2,3,4,5]); heights={None:0}; rows=[]
for node in _postorder_nodes(r):
    left,right=heights[node.left],heights[node.right]
    heights[node]=1+max(left,right); rows.append((node.val,left,right,heights[node],left+right))
show_table(['节点','左高','右高','本树高','经过本点的直径候选'],rows)

节点,左高,右高,本树高,经过本点的直径候选
4,0,0,1,0
5,0,0,1,0
2,1,1,2,2
3,0,0,1,0
1,2,1,3,3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert diameter(TreeNode(1))==0
r=TreeNode(1,TreeNode(2,TreeNode(3))); assert diameter(r)==2 and not is_balanced(r)
assert max_depth(None)==0 and is_balanced(None)
from collections import deque
from random import Random
rng=Random(71)
for n in range(1,30):
    nodes=[TreeNode(i) for i in range(n)]; slots=[(nodes[0],'left'),(nodes[0],'right')]
    adj={x:[] for x in nodes}
    for node in nodes[1:]:
        parent,side=slots.pop(rng.randrange(len(slots))); setattr(parent,side,node)
        slots.extend([(node,'left'),(node,'right')]); adj[parent].append(node); adj[node].append(parent)
    longest=0
    for start in nodes:
        q=deque([(start,0)]); seen={start}
        while q:
            u,d=q.popleft(); longest=max(longest,d)
            for v in adj[u]:
                if v not in seen: seen.add(v); q.append((v,d+1))
    assert diameter(nodes[0])==longest
print('N071: 所有本章断言通过')

N071: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 将平衡判断从重复高度计算改为一次遍历。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 区分按边计与按点计直径。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 104. Maximum Depth of Binary Tree（canonical）
- 110. Balanced Binary Tree（canonical）
- 543. Diameter of Binary Tree（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。